In [1]:
from google.colab import drive

drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/ADS-Airbnb-Project"

print(PROJECT_PATH)

In [ ]:
import os

print(os.listdir(PROJECT_PATH))

In [ ]:
MODEL_DIR = os.path.join(
    PROJECT_PATH,
    "outputs",
    "models"
)

print("Model directory:")
print(MODEL_DIR)

print("\nFiles:")
print(os.listdir(MODEL_DIR))

In [ ]:
import pickle
import os

MODEL_FILE = os.path.join(
    MODEL_DIR,
    "best_airbnb_price_model.pkl"
)

with open(MODEL_FILE, "rb") as file:
    model = pickle.load(file)

print("Model loaded successfully.")
print("Model type:", type(model))

In [ ]:
FEATURE_FILE = os.path.join(
    MODEL_DIR,
    "feature_columns.json"
)

print(
    "Feature file exists:",
    os.path.exists(FEATURE_FILE)
)

In [ ]:
print(model)

In [ ]:
preprocessor = model.named_steps["preprocessor"]

feature_columns = []

for name, transformer, columns in preprocessor.transformers_:
    if columns is not None:
        feature_columns.extend(list(columns))

print("Number of expected input columns:", len(feature_columns))

print("\nExpected input columns:")
for i, column in enumerate(feature_columns, start=1):
    print(i, column)

Step 2 — Check the two groups separately

In [ ]:
print("NUMERICAL FEATURES")
print("=" * 50)

numeric_features = []

for name, transformer, columns in preprocessor.transformers_:
    if name == "num":
        numeric_features = list(columns)

print("Number:", len(numeric_features))
print(numeric_features)

In [ ]:
print("\nCATEGORICAL FEATURES")
print("=" * 50)

categorical_features = []

for name, transformer, columns in preprocessor.transformers_:
    if name == "cat":
        categorical_features = list(columns)

print("Number:", len(categorical_features))
print(categorical_features)

Step 3 — Save feature_columns.json

In [ ]:
import json

FEATURE_FILE = os.path.join(
    MODEL_DIR,
    "feature_columns.json"
)

with open(FEATURE_FILE, "w") as file:
    json.dump(
        feature_columns,
        file,
        indent=4
    )

print("Feature file created successfully.")
print(FEATURE_FILE)

In [ ]:
print(
    "Feature file exists:",
    os.path.exists(FEATURE_FILE)
)

Step 4 — Copy it to the deployment folder

In [ ]:
DEPLOYMENT_DIR = os.path.join(
    PROJECT_PATH,
    "deployment"
)

os.makedirs(
    DEPLOYMENT_DIR,
    exist_ok=True
)

In [ ]:
import shutil

shutil.copy(
    MODEL_FILE,
    os.path.join(
        DEPLOYMENT_DIR,
        "best_airbnb_price_model.pkl"
    )
)

In [ ]:
shutil.copy(
    FEATURE_FILE,
    os.path.join(
        DEPLOYMENT_DIR,
        "feature_columns.json"
    )
)

In [ ]:
print(os.listdir(DEPLOYMENT_DIR))

Step 5 — One more important check

In [ ]:
print("Best model:")
print(model.named_steps["model"])

In [ ]:
print("Pipeline steps:")
print(model.named_steps.keys())

In [ ]:
print("Total input columns:", len(feature_columns))

In [ ]:
print("Numerical columns:", len(numeric_features))
print("Categorical columns:", len(categorical_features))

In [ ]:
print(model.named_steps["model"])

STEP 6 — Create the Deployment Folder

In [ ]:
DEPLOYMENT_DIR = os.path.join(
    PROJECT_PATH,
    "deployment"
)

os.makedirs(
    DEPLOYMENT_DIR,
    exist_ok=True
)

print("Deployment directory:")
print(DEPLOYMENT_DIR)

In [ ]:
import shutil

shutil.copy(
    MODEL_FILE,
    os.path.join(
        DEPLOYMENT_DIR,
        "best_airbnb_price_model.pkl"
    )
)

print("Model copied successfully.")

In [ ]:
shutil.copy(
    FEATURE_FILE,
    os.path.join(
        DEPLOYMENT_DIR,
        "feature_columns.json"
    )
)

print("Feature file copied successfully.")

In [ ]:
print(os.listdir(DEPLOYMENT_DIR))

STEP 7 — Create app.py

In [ ]:
app_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import pickle
import json
import os


# --------------------------------------------------
# FastAPI Application
# --------------------------------------------------

app = FastAPI(
    title="Airbnb Price Prediction API",
    description="API for predicting Airbnb listing prices",
    version="1.0"
)


# --------------------------------------------------
# File Paths
# --------------------------------------------------

BASE_DIR = os.path.dirname(
    os.path.abspath(__file__)
)

MODEL_PATH = os.path.join(
    BASE_DIR,
    "best_airbnb_price_model.pkl"
)

FEATURE_PATH = os.path.join(
    BASE_DIR,
    "feature_columns.json"
)


# --------------------------------------------------
# Load Trained Model
# --------------------------------------------------

with open(MODEL_PATH, "rb") as file:
    model = pickle.load(file)


# --------------------------------------------------
# Load Feature Columns
# --------------------------------------------------

with open(FEATURE_PATH, "r") as file:
    feature_columns = json.load(file)


# --------------------------------------------------
# Request Schema
# --------------------------------------------------

class PredictionRequest(BaseModel):
    data: dict


# --------------------------------------------------
# Root Endpoint
# --------------------------------------------------

@app.get("/")
def home():

    return {
        "message": "Airbnb Price Prediction API is running",
        "endpoint": "/predict",
        "model": "Gradient Boosting Regressor"
    }


# --------------------------------------------------
# Prediction Endpoint
# --------------------------------------------------

@app.post("/predict")
def predict(request: PredictionRequest):

    try:

        # Receive input JSON
        input_data = request.data

        # Convert JSON into DataFrame
        input_df = pd.DataFrame(
            [input_data]
        )

        # Add missing expected columns
        for column in feature_columns:

            if column not in input_df.columns:
                input_df[column] = None

        # Keep exactly the columns used during training
        input_df = input_df[
            feature_columns
        ]

        # Generate prediction
        prediction = model.predict(
            input_df
        )

        predicted_price = float(
            prediction[0]
        )

        return {
            "predicted_price": round(
                predicted_price,
                2
            )
        }

    except Exception as e:

        raise HTTPException(
            status_code=400,
            detail=str(e)
        )
'''

APP_FILE = os.path.join(
    DEPLOYMENT_DIR,
    "app.py"
)

with open(APP_FILE, "w") as file:
    file.write(app_code)

print("app.py created successfully.")

STEP 8 — Verify app.py

In [ ]:
print(os.listdir(DEPLOYMENT_DIR))

In [ ]:
with open(APP_FILE, "r") as file:
    print(file.read())

STEP 9 — Install FastAPI

In [ ]:
!pip install -q fastapi uvicorn

In [ ]:
import fastapi
import uvicorn

print("FastAPI:", fastapi.__version__)
print("Uvicorn installed successfully.")

STEP 10 — Create requirements.txt

In [ ]:
requirements = """fastapi
uvicorn[standard]
pandas
numpy
scikit-learn
pydantic
"""

REQUIREMENTS_FILE = os.path.join(
    DEPLOYMENT_DIR,
    "requirements.txt"
)

with open(
    REQUIREMENTS_FILE,
    "w"
) as file:
    file.write(requirements)

print("requirements.txt created.")

STEP 11 — We need a REAL sample request

In [ ]:
PROCESSED_DIR = os.path.join(
    PROJECT_PATH,
    "datasets",
    "processed"
)

print(os.listdir(PROCESSED_DIR))

In [ ]:
MERGED_FILE = os.path.join(
    PROCESSED_DIR,
    "merged_dataset.csv"
)

merged_df = pd.read_csv(
    MERGED_FILE
)

print("Merged dataset shape:", merged_df.shape)

In [ ]:
import pandas as pd
import numpy as np
import os
import json

In [ ]:
MERGED_FILE = os.path.join(
    PROCESSED_DIR,
    "merged_dataset.csv"
)

merged_df = pd.read_csv(
    MERGED_FILE
)

print("Merged dataset shape:", merged_df.shape)

STEP 12 — Create the API sample from a real listing

In [ ]:
sample_row = merged_df.iloc[0]

sample_input = {}

for column in feature_columns:

    if column in merged_df.columns:

        value = sample_row[column]

        if pd.isna(value):
            sample_input[column] = None

        elif isinstance(value, np.integer):
            sample_input[column] = int(value)

        elif isinstance(value, np.floating):
            sample_input[column] = float(value)

        elif isinstance(value, np.bool_):
            sample_input[column] = bool(value)

        else:
            sample_input[column] = value

    else:
        sample_input[column] = None

In [ ]:
print("Number of input fields:", len(sample_input))

STEP 13 — Create sample_request.json

In [ ]:
sample_request = {
    "data": sample_input
}

SAMPLE_FILE = os.path.join(
    DEPLOYMENT_DIR,
    "sample_request.json"
)

with open(
    SAMPLE_FILE,
    "w"
) as file:

    json.dump(
        sample_request,
        file,
        indent=4,
        default=str
    )

print("sample_request.json created.")

In [ ]:
print(os.listdir(DEPLOYMENT_DIR))

STEP 14 — Test the Model Before Starting the API

In [ ]:
test_input_df = pd.DataFrame(
    [sample_input]
)

test_input_df = test_input_df[
    feature_columns
]

test_prediction = model.predict(
    test_input_df
)

print("Test prediction:", test_prediction[0])

STEP 15 — Start FastAPI in Colab

In [ ]:
%cd "$DEPLOYMENT_DIR"

In [ ]:
!ls -lh

In [ ]:
!nohup uvicorn app:app --host 0.0.0.0 --port 8000 > /tmp/fastapi.log 2>&1 &

In [ ]:
!cat /tmp/fastapi.log

STEP 16 — Test the Root API

In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/"
)

print("Status Code:", response.status_code)
print("Response:")
print(response.json())

STEP 17 — Test /predict

In [ ]:
with open(
    SAMPLE_FILE,
    "r"
) as file:

    sample_request = json.load(file)

In [ ]:
response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=sample_request
)

print("Status Code:", response.status_code)
print("Response:")
print(response.json())

STEP 18 — Create the Dockerfile

In [ ]:
dockerfile = """FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY best_airbnb_price_model.pkl .
COPY feature_columns.json .
COPY sample_request.json .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

DOCKERFILE_PATH = os.path.join(
    DEPLOYMENT_DIR,
    "Dockerfile"
)

with open(DOCKERFILE_PATH, "w") as file:
    file.write(dockerfile)

print("Dockerfile created successfully.")

In [ ]:
!ls -lh

STEP 19 — Understand the Dockerfile

In [ ]:
FROM python:3.11-slim

Verify that the Dockerfile was actually created

In [ ]:
DOCKERFILE_PATH = os.path.join(
    DEPLOYMENT_DIR,
    "Dockerfile"
)

print("Dockerfile exists:",
      os.path.exists(DOCKERFILE_PATH))

In [ ]:
with open(DOCKERFILE_PATH, "r") as file:
    print(file.read())